In [21]:
# use langchain and openai key to generate recommendations based on SHAP impactful features
# import necessary libraries
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_openai import ChatOpenAI
from prompts import prompt1
import os
from dotenv import load_dotenv

load_dotenv()

True

In [18]:

# Initialize model (GPT-4o-mini is fast and cheap)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# Generate a completion
response = llm.invoke("Write a short limerick about football strategy. Keep it under 100 words.")
print(response.content)


In a huddle, the coach lays the plan,  
With a wink and a nod, they all ran.  
A fake to the right,  
Then a dash to the left,  
In the end zone, they score—what a span!  


In [10]:
# parent directory
os.path.abspath(os.path.join(os.getcwd(), '..'))

'/mnt/d/Projects/AIcoach/src'

In [11]:
# add src to the path
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [15]:
from prompts import prompt1, prompt_opponent

In [10]:
# read json files for home and away impactful features
import numpy as np
import pandas as pd
df = pd.read_csv("player_evaluation_results.csv")


In [11]:
df[["name", "pred_rating", "weakness"]].iloc[0]



name                                Ben White
pred_rating                          5.404655
weakness       ['SoTA', 'Tkl+Int', 'Def 3rd']
Name: 0, dtype: object

In [22]:
input_data_format = []

for _, row in df.iterrows():
    player_name = row["name"]
    rating = row["pred_rating"]
    weakness = row["weakness"]

    # 统一格式为字符串
    player_data = f"Player Name: {player_name}\n"
    player_data += f"  pred_rating: {round(float(rating), 4)}\n"

    # weakness 可能是 list 或字符串
    if isinstance(weakness, (list, tuple)):
        weakness_str = ", ".join(map(str, weakness))
    else:
        weakness_str = str(weakness)
    player_data += f"  weakness: {weakness_str}\n"

    input_data_format.append({
        "player_data": player_data.strip(),
        "name": player_name,
        "pred_rating": round(float(rating), 4),
        "weakness": weakness_str
    })

print(input_data_format[:3])  # 预览前3个结果


[{'player_data': "Player Name: Ben White\n  pred_rating: 5.4047\n  weakness: ['SoTA', 'Tkl+Int', 'Def 3rd']", 'name': 'Ben White', 'pred_rating': 5.4047, 'weakness': "['SoTA', 'Tkl+Int', 'Def 3rd']"}, {'player_data': "Player Name: Bukayo Saka\n  pred_rating: 5.4736\n  weakness: ['Sh', 'Tkl%', 'SoTA']", 'name': 'Bukayo Saka', 'pred_rating': 5.4736, 'weakness': "['Sh', 'Tkl%', 'SoTA']"}, {'player_data': "Player Name: Christian Nørgaard\n  pred_rating: 5.828\n  weakness: ['Lost_inv', 'TklW', 'Def 3rd']", 'name': 'Christian Nørgaard', 'pred_rating': 5.828, 'weakness': "['Lost_inv', 'TklW', 'Def 3rd']"}]


In [23]:

# Initialize model (GPT-4o-mini is fast and cheap)
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

In [26]:
record_match_responses = []
for i in input_data_format:
    # use prompt1 from player_data
    final_prompt = prompt1.format(player_data=i['player_data'])

    # create llm chain
    llm_chain = LLMChain(llm=llm, prompt=prompt1)
    # get response
    response = llm_chain.run(player_data=i['player_data'])

    record_match_responses.append({"name": i['name'], "training_recommendations": response})

In [27]:
import pandas as pd
pd.DataFrame(record_match_responses).to_csv('player_training_recommendations.csv', index=False)

In [28]:
teams_list = [
    "Arsenal",
    "Aston Villa",
    "Bournemouth",
    "Brentford",
    "Brighton",
    "Burnley",
    "Chelsea",
    "Crystal Palace",
    "Everton",
    "Fulham",
    "Leeds United",
    "Liverpool",
    "Manchester City",
    "Manchester Utd",
    "Newcastle Utd",
    "Nott'ham Forest",
    "Sunderland",
    "Tottenham",
    "West Ham",
    "Wolves"
]

In [30]:
import pandas as pd
import json
import os

# 文件路径
recommendations_csv = "player_training_recommendations.csv"
evaluation_csv = "player_evaluation_results.csv"
teams_json = "teams_player_list.json"

# 输出目录
output_dir = "teams_csv"
os.makedirs(output_dir, exist_ok=True)

# 读取数据
df_recommendations = pd.read_csv(recommendations_csv, encoding="utf-8-sig")
df_evaluation = pd.read_csv(evaluation_csv, encoding="utf-8-sig")

# 读取球队-球员列表
with open(teams_json, "r", encoding="utf-8") as f:
    teams_player_list = json.load(f)

# 合并两个 DataFrame
df_merged = pd.merge(df_evaluation, df_recommendations, on="name", how="left")

# 遍历每支球队，生成对应 CSV
for team, players in teams_player_list.items():
    # 保留球队球员顺序
    df_team = df_merged[df_merged["name"].isin(players)].copy()
    df_team["name"] = pd.Categorical(df_team["name"], categories=players, ordered=True)
    df_team = df_team.sort_values("name")
    
    # 重命名列
    df_team = df_team.rename(columns={"record_match_responses": "recommendation"})
    
    # 输出 CSV
    output_path = os.path.join(output_dir, f"{team}_player_recommendation_summary.csv")
    df_team.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ 已生成 {output_path}")


✅ 已生成 teams_csv\Arsenal_player_recommendation_summary.csv
✅ 已生成 teams_csv\Aston Villa_player_recommendation_summary.csv
✅ 已生成 teams_csv\Bournemouth_player_recommendation_summary.csv
✅ 已生成 teams_csv\Brentford_player_recommendation_summary.csv
✅ 已生成 teams_csv\Brighton_player_recommendation_summary.csv
✅ 已生成 teams_csv\Burnley_player_recommendation_summary.csv
✅ 已生成 teams_csv\Chelsea_player_recommendation_summary.csv
✅ 已生成 teams_csv\Crystal Palace_player_recommendation_summary.csv
✅ 已生成 teams_csv\Everton_player_recommendation_summary.csv
✅ 已生成 teams_csv\Fulham_player_recommendation_summary.csv
✅ 已生成 teams_csv\Leeds United_player_recommendation_summary.csv
✅ 已生成 teams_csv\Liverpool_player_recommendation_summary.csv
✅ 已生成 teams_csv\Manchester City_player_recommendation_summary.csv
✅ 已生成 teams_csv\Manchester Utd_player_recommendation_summary.csv
✅ 已生成 teams_csv\Newcastle Utd_player_recommendation_summary.csv
✅ 已生成 teams_csv\Nott'ham Forest_player_recommendation_summary.csv
✅ 已生成 teams_csv\Sun